In [ ]:
# =========================
# Pure plotting from saved CSVs
# Reproduce original layout/style
# =========================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------
# 1. load data
# -------------------------
PROJECT_ROOT = Path("../..").resolve()
DATA_DIR = PROJECT_ROOT /"results"/"final"/"Task1717"/"1717_Ablation"

df_mask_50 = pd.read_csv(DATA_DIR /"df_mask_50_all_ckpt.csv")
df_mean = pd.read_csv(DATA_DIR / "df_mean_delta_curve.csv")
regions = pd.read_csv(DATA_DIR / "df_selected_regions.csv")
df_combo_mean = pd.read_csv(DATA_DIR / "df_combo_mean.csv")

# -------------------------
# 2. helper
# -------------------------
def find_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of these columns found: {candidates}")

# line-plot columns
x_col_mask = find_first_existing(df_mask_50, ["rt_mid", "RT_mid", "rt", "x"])
y_col_mask = find_first_existing(df_mask_50, ["DELTA", "delta", "delta_auc", "y"])

x_col_mean = find_first_existing(df_mean, ["rt_mid", "RT_mid", "rt", "x"])
y_col_mean = find_first_existing(df_mean, ["DELTA", "delta", "delta_auc", "y"])

# region columns
start_col = find_first_existing(regions, ["rt_lo", "rt_start", "xmin"])
end_col   = find_first_existing(regions, ["rt_hi", "rt_end", "xmax"])

region_label_col = None
for c in ["label", "region", "region_label", "name"]:
    if c in regions.columns:
        region_label_col = c
        break

if region_label_col is None:
    regions = regions.copy()
    regions["region_label"] = [f"R{i+1}" for i in range(len(regions))]
    region_label_col = "region_label"

# combo columns
combo_label_col = find_first_existing(df_combo_mean, ["combo_label", "label"])
auc_only_col    = find_first_existing(df_combo_mean, ["AUC_only", "auc_only"])
baseline_col    = find_first_existing(df_combo_mean, ["AUC_base_mask", "auc_base_mask", "AUC_base", "auc_base"])

# important: B panel should use DELTA_mask, not AUC_masked
delta_mask_col  = find_first_existing(df_combo_mean, ["DELTA_mask", "delta_mask"])

# -------------------------
# 3. sorting
# -------------------------
df_mask_50 = df_mask_50.sort_values(x_col_mask).copy()
df_mean = df_mean.sort_values(x_col_mean).copy()
regions = regions.sort_values("region").copy()

desired_order = ["R1", "R2", "R3", "R1+R2", "R1+R3", "R2+R3", "R1+R2+R3"]
df_combo_mean = df_combo_mean.copy()
df_combo_mean[combo_label_col] = df_combo_mean[combo_label_col].astype(str)

available_labels = df_combo_mean[combo_label_col].tolist()
plot_order = [x for x in desired_order if x in available_labels]
if len(plot_order) == 0:
    plot_order = available_labels

df_plot = (
    df_combo_mean
    .set_index(combo_label_col)
    .loc[plot_order]
    .reset_index()
)

baseline_auc = float(df_plot[baseline_col].iloc[0])

# -------------------------
# 4. style
# -------------------------
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

main_color = "#2A5D9F"   # closer to your original blue
thin_alpha = 0.2
shade_alpha = 0.12

# -------------------------
# 5. figure layout (match original)
# -------------------------
fig = plt.figure(figsize=(7.2, 5.2), dpi=300)
gs = GridSpec(2, 2, height_ratios=[1.9, 0.95], hspace=0.45, wspace=0.30)

ax_top = fig.add_subplot(gs[0, :])
ax_b1  = fig.add_subplot(gs[1, 0])
ax_b2  = fig.add_subplot(gs[1, 1])

# -------------------------
# 6. top panel: RT ablation profile
# -------------------------
group_col = None
for c in ["ckpt_name", "seed", "model", "run", "id"]:
    if c in df_mask_50.columns:
        group_col = c
        break

if group_col is not None:
    for _, g in df_mask_50.groupby(group_col):
        g = g.sort_values(x_col_mask)
        ax_top.plot(
            g[x_col_mask].values,
            g[y_col_mask].values,
            color=main_color,
            linewidth=0.8,
            alpha=0.12,
        )
else:
    ax_top.plot(
        df_mask_50[x_col_mask].values,
        df_mask_50[y_col_mask].values,
        color=main_color,
        linewidth=1.2,
        alpha=thin_alpha
    )

# mean curve
ax_top.plot(
    df_mean[x_col_mean].values,
    df_mean[y_col_mean].values,
    color=main_color,
    #marker="o",
    markersize=4,
    linewidth=3.0
)

# zero line
ax_top.axhline(0, color="gray", linewidth=1.0)


ax_top.grid(axis="y", linestyle="--", alpha=0.2)

# hotspot shading
ymax = max(float(df_mask_50[y_col_mask].max()), float(df_mean[y_col_mean].max()))
ymin = min(float(df_mask_50[y_col_mask].min()), float(df_mean[y_col_mean].min()))

yr = ymax - ymin if ymax > ymin else 1.0

for i, (_, row) in enumerate(regions.iterrows()):
    x0 = row[start_col]
    x1 = row[end_col]

    # 灰色区间（全部画）
    ax_top.axvspan(
        x0, x1,
        color="#C9CED6",
        alpha=shade_alpha,
        ymin=0.0,
        ymax=0.85
    )

    # 只标前5个
    if i < 5:
        lab = f"R{i+1}"

        ax_top.text(
            (x0 + x1) / 2,
            0.062,
            lab,
            ha="center",
            va="center",
            fontsize=7.5,
            fontweight="normal"
        )




ax_top.set_xlim(df_mean[x_col_mean].min() - 20, df_mean[x_col_mean].max() + 20)
ax_top.set_ylim(-0.05, 0.11) 
ax_top.set_yticks([-0.05, 0.0, 0.05])

ax_top.set_xlabel("RT (s)")
ax_top.set_ylabel("ΔAUROC")

# overall title + task annotation
ax_top.set_title("RT region identification via ablation (top-30% models)",pad=6, fontsize=12)
ax_top.text(
    0.01, 0.94,
    "D RT importance profile (STP1717.1 vs control)",
    transform=ax_top.transAxes,
    ha="left", va="top",
    fontsize=9.5
)

# -------------------------
# 7. bottom-left: masked-region importance
# -------------------------
x = np.arange(len(df_plot))

ax_b1.bar(
    x,
    df_plot[delta_mask_col].values,
    width=0.78,
    alpha=0.9,
    color="#2A5D9F"
)
ax_b1.set_ylim(0.0, 0.1) 
ax_b1.set_yticks([ 0.0, 0.05,0.1])

ax_b1.set_xticks(x)

labels = [l.replace("+", "+\n") for l in df_plot[combo_label_col]]

#ax_b1.set_xticklabels(df_plot[combo_label_col].tolist(), rotation=20,ha="left")
ax_b1.set_xticklabels(labels, rotation=0)
ax_b1.set_ylabel("ΔAUROC")
ax_b1.set_xlabel("Hotspot combination")
ax_b1.set_title("E Masked-region importance", loc="left", fontsize=9.5)




# -------------------------
# 8. bottom-right: region-only performance
# -------------------------
ax_b2.bar(
    x,
    df_plot[auc_only_col].values,
    width=0.78,
    alpha=0.9,
    color="#2A5D9F"
)
ax_b2.axhline(baseline_auc, color="#C7C7C7", linestyle="--", linewidth=0.9)
ax_b2.text(
    0.8, baseline_auc+0.002,
    "baseline",
    transform=ax_b2.get_yaxis_transform(),
    ha="left", va="bottom",
    color="#2F6FB3",
    fontsize=8
)
ax_b2.set_ylim(0.0, 1.0) 
ax_b2.set_yticks([ 0.0, 0.5,1.0])
ax_b2.set_xticks(x)
#labels = [l.replace("+", "+\n") for l in df_plot[combo_label_col]]
ax_b2.set_xticklabels(labels, rotation=0)
#ax_b2.set_xticklabels(df_plot[combo_label_col].tolist(), rotation=20,ha="left")
ax_b2.set_ylabel("AUROC")
ax_b2.set_xlabel("Hotspot combination")
ax_b2.set_title("F Region-only performance", loc="left", fontsize=9.5)

ax_b2.axhline(
    baseline_auc,
    linestyle="--",
    color="#B5B5B5",
    label="baseline"
)

# ax_b2.legend(loc="upper right", frameon=False)

# -------------------------
# 9. clean up
# -------------------------
for ax in [ax_top, ax_b1, ax_b2]:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.97])


OUT_DIR =  PROJECT_ROOT / "figures/qc"

plt.savefig(OUT_DIR/"ablation_LR_1717.png", dpi=300, bbox_inches="tight")



plt.show()